<a href="https://colab.research.google.com/github/xeunnie/GDG-HandsOn/blob/main/GDG_HandsOn_Colab_ipynb%EC%9D%98_%EC%82%AC%EB%B3%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Colab 런타임에 실습용 저장소를 내려받습니다.
!git clone https://github.com/skqorrla/GDG-HandsOn.git

Cloning into 'GDG-HandsOn'...
remote: Enumerating objects: 22, done.
remote: Counting objects: 100% (22/22), done.
remote: Compressing objects: 100% (20/20), done.
remote: Total 22 (delta 1), reused 22 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (22/22), 126.91 KiB | 1.21 MiB/s, done.
Resolving deltas: 100% (1/1), done.


In [ ]:
# 방금 clone한 저장소 폴더로 작업 디렉터리를 이동합니다.
import os
from pathlib import Path

REPO_NAME = "GDG-HandsOn"
if Path(REPO_NAME).exists():
    os.chdir(REPO_NAME)

In [ ]:
# 벡터 검색 저장소로 사용할 ChromaDB를 설치합니다.
!pip install -q chromadb

# GDG 계층적 메모리 핸즈온

## 1. 환경 설정

- 입력 파일, 생성 JSON 파일, 전용 SQLiteDB, 전용 Chroma 저장소, Gemini 모델, Gemini API Key 설정

In [ ]:
# 실습 데이터와 생성 산출물을 저장할 경로를 한 곳에서 정의합니다.
import os
from pathlib import Path

from dotenv import load_dotenv

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data" / "demo_memory_stm.json").exists():
            return candidate
    raise FileNotFoundError("Could not find repository root containing data/demo_memory_stm.json")

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
# load_dotenv(PROJECT_ROOT / ".env")

INPUT_STM_PATH = PROJECT_ROOT / "data" / "demo_memory_stm.json"
GENERATED_LTM_PATH = PROJECT_ROOT / "data" / "generated_memory_ltm.json"
GENERATED_EPI_PATH = PROJECT_ROOT / "data" / "generated_memory_epi.json"
SQLITE_DB_PATH = PROJECT_ROOT / "data" / "gemini_handson.db"
CHROMA_STORE_PATH = PROJECT_ROOT / "data" / "chroma_gemini_handson"
PROMOTION_MODEL = "gemini-2.5-flash"
CHATBOT_MODEL = "gemini-2.5-flash"
EMBEDDING_MODEL = "gemini-embedding-001"

##--------------반드시 입력해주세요!--------------------
GEMINI_API_KEY = "YOUR_API_KEY"
##------------------------------------------------------

for path in [GENERATED_LTM_PATH, GENERATED_EPI_PATH, SQLITE_DB_PATH]:
    path.parent.mkdir(parents=True, exist_ok=True)
CHROMA_STORE_PATH.mkdir(parents=True, exist_ok=True)
if not INPUT_STM_PATH.exists():
    raise FileNotFoundError(INPUT_STM_PATH)

CONFIG = {
    "input_stm_path": INPUT_STM_PATH,
    "generated_ltm_path": GENERATED_LTM_PATH,
    "generated_epi_path": GENERATED_EPI_PATH,
    "sqlite_db_path": SQLITE_DB_PATH,
    "chroma_store_path": CHROMA_STORE_PATH,
    "promotion_model": PROMOTION_MODEL,
    "chatbot_model": CHATBOT_MODEL,
    "gemini_api_key_loaded": bool(GEMINI_API_KEY),
}
CONFIG


{'input_stm_path': PosixPath('/content/GDG-HandsOn/GDG-HandsOn/GDG-HandsOn/data/demo_memory_stm.json'),
 'generated_ltm_path': PosixPath('/content/GDG-HandsOn/GDG-HandsOn/GDG-HandsOn/data/generated_memory_ltm.json'),
 'generated_epi_path': PosixPath('/content/GDG-HandsOn/GDG-HandsOn/GDG-HandsOn/data/generated_memory_epi.json'),
 'sqlite_db_path': PosixPath('/content/GDG-HandsOn/GDG-HandsOn/GDG-HandsOn/data/gemini_handson.db'),
 'chroma_store_path': PosixPath('/content/GDG-HandsOn/GDG-HandsOn/GDG-HandsOn/data/chroma_gemini_handson'),
 'promotion_model': 'gemini-2.5-flash',
 'chatbot_model': 'gemini-2.5-flash',
 'gemini_api_key_loaded': True}

## 2. STM 입력 확인

- `data/demo_memory_stm.json` : 데모 데이터
    - `stm_conversations`: 여러 STM 세션을 담은 최상위 배열입니다.
        - `session_id`: 하나의 대화 세션을 식별하는 ID입니다.
        - `recent_topic`: 해당 세션의 최근/대표 주제를 나타내는 태그입니다.
        - `messages`: 세션 안에서 오간 raw 대화 메시지 배열입니다.
            - `id`: 각 메시지를 식별하는 고유 ID입니다.
            - `memory_type`: 메모리 계층을 나타내는 값이며, STM 원본에서는 `stm`입니다.
            - `role`: 메시지 발신자입니다. `user` 또는 `assistant` 값을 가집니다.
            - `content`: 실제 대화 메시지 원문입니다.
            - `timestamp`: 메시지가 생성된 시각입니다. ISO 8601 형식으로 저장됩니다.
            - `turn_index`: 세션 안에서 메시지가 몇 번째 turn인지 나타내는 순서값입니다.

### 데모 데이터 페르소나
- 수준: 파이썬 기초를 배우는 초급 학습자
- 학습 주제: 반복문, 조건문, 함수, 예외 처리, 모듈, 클래스
- 강점: 질문이 구체적이고, 예제로 개념을 확인하려고 함
- 약점: 비슷한 개념의 차이를 자주 헷갈림
- 반복 혼동:
    - `for`와 `if`를 함께 쓰는 흐름
    - `if` 여러 개와 `elif`의 차이
    - `range()`의 시작/끝 범위
    - parameter와 argument
    - `return`과 `print`
    - `try-except`의 사용 범위
    - `import` 방식 차이
    - `class`, `self`, 인스턴스 변수
- 챗봇 응답 전략:
    - 쉬운 예제로 짧게 설명
    - 이전에 헷갈린 개념을 먼저 짚어줌
    - 새 개념을 이전 주제와 연결
    - “네가 전에 헷갈려 했던 부분은...”처럼 메모리 기반 피드백 제공


In [ ]:
# 입력 STM JSON 구조를 검증하고, 실습에서 사용할 대화 개수를 확인합니다.
import json

try:
    from IPython.display import Markdown, display
except ModuleNotFoundError:
    Markdown = str
    display = print


def validate_stm_payload(payload):
    required_conversation_keys = {"session_id", "recent_topic", "messages"}
    required_message_keys = {"id", "memory_type", "session_id", "role", "content", "timestamp", "turn_index"}
    conversations = payload.get("stm_conversations")
    if not isinstance(conversations, list) or not conversations:
        raise ValueError("stm_conversations는 비어 있지 않은 리스트여야 합니다.")

    message_count = 0
    for conversation_index, conversation in enumerate(conversations):
        missing = required_conversation_keys - conversation.keys()
        if missing:
            raise ValueError(f"conversation[{conversation_index}] 누락 필드: {sorted(missing)}")
        if not isinstance(conversation.get("messages"), list) or not conversation["messages"]:
            raise ValueError(f"conversation[{conversation_index}].messages는 비어 있지 않은 리스트여야 합니다.")
        for message_index, message in enumerate(conversation["messages"]):
            missing = required_message_keys - message.keys()
            if missing:
                raise ValueError(f"message[{conversation_index}:{message_index}] 누락 필드: {sorted(missing)}")
            if message.get("memory_type") != "stm":
                raise ValueError(f"message[{conversation_index}:{message_index}] memory_type은 stm이어야 합니다.")
            if message.get("session_id") != conversation["session_id"]:
                raise ValueError(f"message[{conversation_index}:{message_index}] session_id가 대화와 다릅니다.")
            message_count += 1
    return {"session_count": len(conversations), "message_count": message_count}


stm_data = json.loads(INPUT_STM_PATH.read_text(encoding="utf-8"))
validation_summary = validate_stm_payload(stm_data)
stm_conversations = stm_data["stm_conversations"]
message_count = validation_summary["message_count"]

LTM_MEMORY_SCHEMA = {
    "ltm_memory": [
        {
            "id": "string",
            "session_id": "string",
            "summary": "string",
            "struggles": ["string"],
            "strengths": ["string"],
            "confusions": ["string"],
            "topic_tags": ["string"],
            "source_message_ids": ["string"],
            "source_turn_indices": ["integer"],
        }
    ]
}

display(Markdown(
    f"**STM 입력 검증 완료**: `{INPUT_STM_PATH.relative_to(PROJECT_ROOT)}`에서 "
    f"세션 {validation_summary['session_count']}개, 메시지 {message_count}개를 읽고 필수 구조를 확인했습니다."
))
display(validation_summary)
display(Markdown("### 목표 LTM JSON 스키마"))
display(LTM_MEMORY_SCHEMA)
display(Markdown("### STM 데이터"))
display(stm_data)


**STM 입력 검증 완료**: `data/demo_memory_stm.json`에서 세션 6개, 메시지 84개를 읽고 필수 구조를 확인했습니다.

{'session_count': 6, 'message_count': 84}

### 목표 LTM JSON 스키마

{'ltm_memory': [{'id': 'string',
   'session_id': 'string',
   'summary': 'string',
   'struggles': ['string'],
   'strengths': ['string'],
   'confusions': ['string'],
   'topic_tags': ['string'],
   'source_message_ids': ['string'],
   'source_turn_indices': ['integer']}]}

### STM 데이터

{'stm_conversations': [{'session_id': 'demo-stm-20260501-loops-conditionals',
   'recent_topic': 'programming:python-loops-conditionals',
   'messages': [{'id': 'demo-stm-20260501-loops-conditionals-000',
     'memory_type': 'stm',
     'session_id': 'demo-stm-20260501-loops-conditionals',
     'role': 'user',
     'content': 'for문이랑 if문을 같이 쓰는 게 헷갈려. 리스트에서 짝수만 출력하려면 먼저 반복하고 안에서 조건을 보는 거야?',
     'timestamp': '2026-05-01T11:00:00+09:00',
     'turn_index': 0},
    {'id': 'demo-stm-20260501-loops-conditionals-001',
     'memory_type': 'stm',
     'session_id': 'demo-stm-20260501-loops-conditionals',
     'role': 'assistant',
     'content': '맞아. for number in numbers로 하나씩 꺼내고, 반복문 안에서 if number % 2 == 0 조건을 검사하면 짝수일 때만 실행할 수 있어.',
     'timestamp': '2026-05-01T11:00:34+09:00',
     'turn_index': 1},
    {'id': 'demo-stm-20260501-loops-conditionals-002',
     'memory_type': 'stm',
     'session_id': 'demo-stm-20260501-loops-conditionals',
     'role': 'user',
     'content': '그러면 if를 for

## 3. STM -> LTM 변환

- 변환 기준: 3시간 이상 새로운 대화가 진행되지 않을 때
- `gemini-2.5-pro`(핸즈온에서는 `gemini-2.5-flash`)를 호출해서 **학습 패턴을 요약·구조화**

- `data/generated_memory_ltm.json` : STM 대화 기록을 Gemini로 요약해 만든 LTM 장기 기억 데이터
    - `id`: 장기 기억 요약을 추적하는 고유 ID입니다.
    - `session_id`: 어떤 STM 세션에서 만들어졌는지 가리키는 provenance 키입니다.
    - `summary`: 여러 raw turn을 한 문단으로 압축한 장기 기억 요약입니다.
    - `struggles`: 학습자가 어려워한 개념이나 패턴의 JSON 배열입니다.
    - `strengths`: 학습자가 잘 수행한 점의 JSON 배열입니다.
    - `confusions`: 아직 남아 있는 오해나 확인이 필요한 질문의 JSON 배열입니다.
    - `topic_tags`: 검색과 Episodic 집계를 위한 topic 태그 JSON 배열입니다.
    - `source_message_ids`: 이 LTM 요약을 만들 때 근거로 사용된 STM 메시지 ID 배열입니다.
    - `source_turn_indices`: 이 LTM 요약을 만들 때 근거로 사용된 STM 메시지의 세션 내 turn 순서 배열입니다.

In [ ]:
from google import genai

if not GEMINI_API_KEY:
    raise RuntimeError(".env에 GEMINI_API_KEY 또는 GOOGLE_API_KEY를 설정하세요.")

client = genai.Client(api_key=GEMINI_API_KEY)

LTM_PROMOTION_INSTRUCTIONS = """
당신은 학습 대화 STM을 장기 기억 JSON으로 승격하는 메모리 정리자입니다.
반드시 JSON만 반환하고, 각 LTM 항목은 원본 session_id, source_message_ids, source_turn_indices를 보존하세요.
summary는 장기적으로 재사용할 학습 맥락을 한국어 한두 문장으로 요약하세요.
struggles, strengths, confusions, topic_tags는 검색과 챗봇 응답에 바로 쓸 수 있는 짧은 한국어 배열로 작성하세요.
""".strip()

def build_ltm_promotion_prompt(stm_payload):
    return f"""
{LTM_PROMOTION_INSTRUCTIONS}

[출력 스키마]
{json.dumps(LTM_MEMORY_SCHEMA, ensure_ascii=False, indent=2)}

[입력 STM]
{json.dumps(stm_payload, ensure_ascii=False, indent=2)}
""".strip()

def parse_gemini_json(response):
    text = getattr(response, "text", None) or response.candidates[0].content.parts[0].text
    text = text.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    return json.loads(text)

# llm payload 검증 함수
def validate_ltm_payload(payload):
    required_ltm_keys = {"id", "session_id", "summary", "struggles", "strengths", "confusions", "topic_tags", "source_message_ids", "source_turn_indices"}
    memories = payload.get("ltm_memory")
    if not isinstance(memories, list) or not memories:
        raise ValueError("ltm_memory는 비어 있지 않은 리스트여야 합니다.")
    stm_session_ids = {conversation["session_id"] for conversation in stm_conversations}
    stm_message_ids = {message["id"] for conversation in stm_conversations for message in conversation["messages"]}
    for index, item in enumerate(memories):
        missing = required_ltm_keys - item.keys()
        if missing:
            raise ValueError(f"ltm_memory[{index}] 누락 필드: {sorted(missing)}")
        if item["session_id"] not in stm_session_ids:
            raise ValueError(f"ltm_memory[{index}] session_id가 STM 원본에 없습니다.")
        if not str(item["summary"]).strip():
            raise ValueError(f"ltm_memory[{index}] summary는 비어 있을 수 없습니다.")
        for field in ["struggles", "strengths", "confusions", "topic_tags", "source_message_ids"]:
            if not isinstance(item[field], list) or not all(isinstance(value, str) for value in item[field]):
                raise ValueError(f"ltm_memory[{index}].{field}는 문자열 리스트여야 합니다.")
        if not item["source_message_ids"] or not set(item["source_message_ids"]).issubset(stm_message_ids):
            raise ValueError(f"ltm_memory[{index}] source_message_ids가 STM 원본과 맞지 않습니다.")
        if not isinstance(item["source_turn_indices"], list) or not all(isinstance(value, int) for value in item["source_turn_indices"]):
            raise ValueError(f"ltm_memory[{index}].source_turn_indices는 정수 리스트여야 합니다.")
    return payload

def validate_generated_ltm_json_structure(payload):
    required_ltm_keys = ["id", "session_id", "summary", "struggles", "strengths", "confusions", "topic_tags", "source_message_ids", "source_turn_indices"]
    memories = payload["ltm_memory"]
    return {
        "root_key_present": "ltm_memory" in payload,
        "ltm_memory_type": type(memories).__name__,
        "ltm_count": len(memories),
        "required_fields": required_ltm_keys,
        "field_presence_by_item": [
            {field: field in item for field in required_ltm_keys}
            for item in memories
        ],
    }

def promote_stm_with_gemini(stm_payload):
    if stm_payload != stm_data:
        raise ValueError("이 핸즈온 셀은 위에서 검증한 STM 입력만 승격합니다.")
    response = client.models.generate_content(model=PROMOTION_MODEL,
                                              contents=ltm_promotion_prompt,
                                              config={"response_mime_type": "application/json"})
    return parse_gemini_json(response)

# 프롬프트 구성
ltm_promotion_prompt = build_ltm_promotion_prompt(stm_data)

# 검증
generated_ltm_memory = validate_ltm_payload(promote_stm_with_gemini(stm_data))
ltm_required_field_report = validate_generated_ltm_json_structure(generated_ltm_memory)
ltm_validation_summary = {"ltm_count": len(generated_ltm_memory["ltm_memory"]), "source": INPUT_STM_PATH.relative_to(PROJECT_ROOT).as_posix(), "model": PROMOTION_MODEL}
ltm_validation_summary["required_fields"] = ltm_required_field_report["required_fields"]

# 출력
display(Markdown("### Gemini LTM 승격 프롬프트"))
display(ltm_promotion_prompt[:4000])
display(Markdown("### Gemini 생성 LTM 메모리"))
display(ltm_validation_summary)
display(Markdown("### LTM JSON 구조 검증"))
display(ltm_required_field_report)
display(Markdown("### 변환된 LTM JSON"))
display(generated_ltm_memory)


### Gemini LTM 승격 프롬프트

'당신은 학습 대화 STM을 장기 기억 JSON으로 승격하는 메모리 정리자입니다.\n반드시 JSON만 반환하고, 각 LTM 항목은 원본 session_id, source_message_ids, source_turn_indices를 보존하세요.\nsummary는 장기적으로 재사용할 학습 맥락을 한국어 한두 문장으로 요약하세요.\nstruggles, strengths, confusions, topic_tags는 검색과 챗봇 응답에 바로 쓸 수 있는 짧은 한국어 배열로 작성하세요.\n\n[출력 스키마]\n{\n  "ltm_memory": [\n    {\n      "id": "string",\n      "session_id": "string",\n      "summary": "string",\n      "struggles": [\n        "string"\n      ],\n      "strengths": [\n        "string"\n      ],\n      "confusions": [\n        "string"\n      ],\n      "topic_tags": [\n        "string"\n      ],\n      "source_message_ids": [\n        "string"\n      ],\n      "source_turn_indices": [\n        "integer"\n      ]\n    }\n  ]\n}\n\n[입력 STM]\n{\n  "stm_conversations": [\n    {\n      "session_id": "demo-stm-20260501-loops-conditionals",\n      "recent_topic": "programming:python-loops-conditionals",\n      "messages": [\n        {\n          "id": "demo-stm-20260501-loops-conditionals-000",\n     

### Gemini 생성 LTM 메모리

{'ltm_count': 6,
 'source': 'data/demo_memory_stm.json',
 'model': 'gemini-2.5-flash',
 'required_fields': ['id',
  'session_id',
  'summary',
  'struggles',
  'strengths',
  'confusions',
  'topic_tags',
  'source_message_ids',
  'source_turn_indices']}

### LTM JSON 구조 검증

{'root_key_present': True,
 'ltm_memory_type': 'list',
 'ltm_count': 6,
 'required_fields': ['id',
  'session_id',
  'summary',
  'struggles',
  'strengths',
  'confusions',
  'topic_tags',
  'source_message_ids',
  'source_turn_indices'],
 'field_presence_by_item': [{'id': True,
   'session_id': True,
   'summary': True,
   'struggles': True,
   'strengths': True,
   'confusions': True,
   'topic_tags': True,
   'source_message_ids': True,
   'source_turn_indices': True},
  {'id': True,
   'session_id': True,
   'summary': True,
   'struggles': True,
   'strengths': True,
   'confusions': True,
   'topic_tags': True,
   'source_message_ids': True,
   'source_turn_indices': True},
  {'id': True,
   'session_id': True,
   'summary': True,
   'struggles': True,
   'strengths': True,
   'confusions': True,
   'topic_tags': True,
   'source_message_ids': True,
   'source_turn_indices': True},
  {'id': True,
   'session_id': True,
   'summary': True,
   'struggles': True,
   'strengths': Tr

{'ltm_memory': [{'id': 'ltm-demo-20260501-loops-conditionals',
   'session_id': 'demo-stm-20260501-loops-conditionals',
   'summary': '파이썬에서 for와 while 반복문, if/elif/else 조건문을 함께 사용하는 방법, 각 문의 역할과 들여쓰기의 중요성, 그리고 break/continue의 차이를 학습했습니다.',
   'struggles': ['for문과 if문을 같이 쓰는 방법',
    'if문의 위치(for 안/밖)에 따른 차이',
    'if를 여러 번 쓰는 것과 elif의 차이',
    'for와 while 반복문 구분',
    '중첩 if/for문의 복잡성'],
   'strengths': ['반복문과 조건문 결합 아이디어 제시',
    'if문의 들여쓰기 역할 이해',
    '리스트에서 특정 조건으로 요소 모으기 적용',
    'elif를 이용한 등급 분류 활용',
    'break와 continue의 역할 구분',
    '핵심 개념을 정확히 요약'],
   'confusions': ['for문과 if문의 결합 순서',
    'if문 들여쓰기의 의미',
    'elif와 독립적인 if 조건 검사의 결과 차이',
    'for와 while 사용 시기',
    'continue와 break의 정확한 동작 차이',
    '중첩 구조의 이해'],
   'topic_tags': ['파이썬',
    '반복문',
    '조건문',
    'for문',
    'if문',
    'elif문',
    'else문',
    'while문',
    'break',
    'continue',
    '들여쓰기',
    '중첩'],
   'source_message_ids': ['demo-stm-20260501-loops-conditionals-000',
    'demo-stm-20260501-loops-conditi

## 4. LTM -> Episodic Memory(일화 기억) 변환

- 변환 기준: 매일 03시
- `gemini-2.5-pro`(핸즈온에서는 `gemini-2.5-flash`)를 호출해서 **같은 주제에서 반복되는 강점·약점·질문 패턴
  을 누적한 기억**
- `data/generated_memory_epi.json` : LTM 장기 기억을 주제 단위로 다시 묶어 만든 Episodic memory 데이터
    - `episodic_id`: 주제별 에피소드 메모리를 식별하는 고유 ID입니다.
    - `topic`: LTM에서 추출된 학습 주제 이름입니다.
    - `topic_tags`: 검색과 분류에 사용할 topic 태그 JSON 배열입니다.
    - `strengths`: 해당 주제에서 학습자가 보인 강점 배열입니다.
    - `weaknesses`: 해당 주제에서 반복적으로 드러난 약점이나 어려움 배열입니다.
    - `questions`: 해당 주제와 관련해 학습자가 실제로 했거나 남긴 질문 배열입니다.
    - `source_session_ids`: 이 Episodic memory를 구성하는 데 사용된 원본 STM 세션 ID 배열입니다.
    - `source_message_ids`: 이 Episodic memory의 근거가 된 원본 STM 메시지 ID 배열입니다.
    - `source_turn_indices`: 근거 메시지들의 세션 내 turn 순서 배열입니다.
    - `source_message_timestamps`: 근거 메시지들이 생성된 시각 배열입니다.
    - `topic_contexts`: 어떤 LTM/세션/요약에서 이 주제가 만들어졌는지 담는 근거 컨텍스트 배열입니다.
    - `occurrence_count`: 이 주제가 몇 번 관찰되거나 병합되었는지 나타내는 횟수입니다.
    - `memory_item_type`: 메모리 항목의 유형입니다. 보통 `learning_event` 값을 사용합니다.
    - `last_updated`: 이 Episodic memory가 마지막으로 생성되거나 갱신된 시각입니다.

In [ ]:
from episodic_schema import EPISODIC_MEMORY_ITEM_TYPES, EPISODIC_REQUIRED_STORAGE_FIELDS

EPISODIC_MEMORY_SCHEMA = {
    "episodic_memory": [
        {
            "episodic_id": "string",
            "topic": "string",
            "topic_tags": ["string"],
            "strengths": ["string"],
            "weaknesses": ["string"],
            "questions": ["string"],
            "source_session_ids": ["string"],
            "source_message_ids": ["string"],
            "source_turn_indices": ["integer"],
            "source_message_timestamps": ["string"],
            "topic_contexts": ["string"],
            "occurrence_count": 1,
            "memory_item_type": "learning_event",
            "last_updated": "ISO-8601 string",
        }
    ]
}

EPISODIC_PROMOTION_INSTRUCTIONS = """
당신은 LTM 학습 기억을 주제 단위 에피소드 메모리 JSON으로 승격하는 메모리 정리자입니다.
반드시 JSON만 반환하고, 각 에피소드 항목은 원본 source_session_ids, source_message_ids, source_turn_indices, source_message_timestamps를 반드시 보존하세요.
topic은 학습 사건의 핵심 주제로, topic_contexts는 이후 챗봇이 참고할 수 있는 짧은 한국어 맥락으로 작성하세요.
strengths, weaknesses, questions, topic_tags는 검색과 피드백 생성에 바로 쓸 수 있는 짧은 한국어 배열로 작성하고, occurrence_count와 last_updated도 채우세요.
""".strip()

def build_episodic_promotion_prompt(ltm_payload):
    return f"""
{EPISODIC_PROMOTION_INSTRUCTIONS}

[출력 스키마]
{json.dumps(EPISODIC_MEMORY_SCHEMA, ensure_ascii=False, indent=2)}

[입력 LTM]
{json.dumps(ltm_payload, ensure_ascii=False, indent=2)}
""".strip()

def promote_ltm_with_gemini(ltm_payload):
    episodic_prompt = build_episodic_promotion_prompt(ltm_payload)
    response = client.models.generate_content(model=PROMOTION_MODEL, contents=episodic_prompt, config={"response_mime_type": "application/json"})
    return parse_gemini_json(response)

def validate_ltm_records_for_episodic(ltm_payload):
    validated_payload = validate_ltm_payload(ltm_payload)
    records = validated_payload["ltm_memory"]
    if not all(record.get("source_message_ids") and record.get("source_turn_indices") for record in records):
        raise ValueError("에피소드 승격 입력 LTM은 source_message_ids와 source_turn_indices를 포함해야 합니다.")
    return records

def validate_episodic_payload(payload):
    records = payload.get("episodic_memory") if isinstance(payload, dict) else None
    if not isinstance(records, list) or not records:
        raise ValueError("episodic_memory는 비어 있지 않은 리스트여야 합니다.")
    for index, record in enumerate(records):
        missing = EPISODIC_REQUIRED_STORAGE_FIELDS - record.keys()
        if missing:
            raise ValueError(f"episodic_memory[{index}] 누락 필드: {sorted(missing)}")
        if record["memory_item_type"] not in EPISODIC_MEMORY_ITEM_TYPES:
            raise ValueError(f"episodic_memory[{index}] memory_item_type 값이 잘못되었습니다.")
    return payload

def validate_generated_episodic_json_structure(payload):
    records = validate_episodic_payload(payload)["episodic_memory"]
    required_fields = sorted(EPISODIC_REQUIRED_STORAGE_FIELDS)
    return {
        "episodic_count": len(records),
        "required_fields": required_fields,
        "records": [{"episodic_id": record["episodic_id"], "present_required_fields": sorted(EPISODIC_REQUIRED_STORAGE_FIELDS & record.keys())} for record in records],
    }

ltm_records_for_episodic = generated_ltm_memory["ltm_memory"]
ltm_source_for_episodic = {"ltm_memory": ltm_records_for_episodic}
ltm_records_for_episodic = validate_ltm_records_for_episodic(ltm_source_for_episodic)
episodic_promotion_prompt = build_episodic_promotion_prompt(ltm_source_for_episodic)
episodic_input_validation_summary = {
    "validated_ltm_record_count": len(ltm_records_for_episodic),
    "source_session_ids": sorted({record["session_id"] for record in ltm_records_for_episodic}),
    "ready_for_episodic_extraction": True,
}

display(Markdown("### 목표 에피소드 JSON 스키마"))
display(EPISODIC_MEMORY_SCHEMA)
display(Markdown("### 에피소드 승격 입력 LTM 레코드"))
display(episodic_input_validation_summary)
display(ltm_records_for_episodic)
display(Markdown("### Gemini 에피소드 승격 프롬프트"))
display(episodic_promotion_prompt[:4000])

generated_episodic_memory = promote_ltm_with_gemini(ltm_source_for_episodic)
generated_episodic_memory = validate_episodic_payload(generated_episodic_memory)
episodic_required_field_report = validate_generated_episodic_json_structure(generated_episodic_memory)
episodic_validation_summary = {"episodic_count": len(generated_episodic_memory["episodic_memory"]), "model": PROMOTION_MODEL}
episodic_validation_summary["required_fields"] = episodic_required_field_report["required_fields"]

display(Markdown("### Gemini 생성 에피소드 메모리"))
display(episodic_validation_summary)
display(Markdown("### 에피소드 JSON 구조 검증"))
display(episodic_required_field_report)
display(generated_episodic_memory)


### 목표 에피소드 JSON 스키마

{'episodic_memory': [{'episodic_id': 'string',
   'topic': 'string',
   'topic_tags': ['string'],
   'strengths': ['string'],
   'weaknesses': ['string'],
   'questions': ['string'],
   'source_session_ids': ['string'],
   'source_message_ids': ['string'],
   'source_turn_indices': ['integer'],
   'source_message_timestamps': ['string'],
   'topic_contexts': ['string'],
   'occurrence_count': 1,
   'memory_item_type': 'learning_event',
   'last_updated': 'ISO-8601 string'}]}

### 에피소드 승격 입력 LTM 레코드

{'validated_ltm_record_count': 6,
 'source_session_ids': ['demo-stm-20260501-functions-exceptions',
  'demo-stm-20260501-loops-conditionals',
  'demo-stm-20260502-modules',
  'demo-stm-20260504-classes',
  'demo-stm-20260507-conditionals',
  'demo-stm-20260507-loops'],
 'ready_for_episodic_extraction': True}

[{'id': 'ltm-demo-20260501-loops-conditionals',
  'session_id': 'demo-stm-20260501-loops-conditionals',
  'summary': '파이썬에서 for와 while 반복문, if/elif/else 조건문을 함께 사용하는 방법, 각 문의 역할과 들여쓰기의 중요성, 그리고 break/continue의 차이를 학습했습니다.',
  'struggles': ['for문과 if문을 같이 쓰는 방법',
   'if문의 위치(for 안/밖)에 따른 차이',
   'if를 여러 번 쓰는 것과 elif의 차이',
   'for와 while 반복문 구분',
   '중첩 if/for문의 복잡성'],
  'strengths': ['반복문과 조건문 결합 아이디어 제시',
   'if문의 들여쓰기 역할 이해',
   '리스트에서 특정 조건으로 요소 모으기 적용',
   'elif를 이용한 등급 분류 활용',
   'break와 continue의 역할 구분',
   '핵심 개념을 정확히 요약'],
  'confusions': ['for문과 if문의 결합 순서',
   'if문 들여쓰기의 의미',
   'elif와 독립적인 if 조건 검사의 결과 차이',
   'for와 while 사용 시기',
   'continue와 break의 정확한 동작 차이',
   '중첩 구조의 이해'],
  'topic_tags': ['파이썬',
   '반복문',
   '조건문',
   'for문',
   'if문',
   'elif문',
   'else문',
   'while문',
   'break',
   'continue',
   '들여쓰기',
   '중첩'],
  'source_message_ids': ['demo-stm-20260501-loops-conditionals-000',
   'demo-stm-20260501-loops-conditionals-001',
   'demo-stm-20260501-loops-conditio

...

### Gemini 생성 에피소드 메모리

{'episodic_count': 6,
 'model': 'gemini-2.5-flash',
 'required_fields': ['episodic_id',
  'last_updated',
  'memory_item_type',
  'occurrence_count',
  'questions',
  'source_message_ids',
  'source_message_timestamps',
  'source_session_ids',
  'source_turn_indices',
  'strengths',
  'topic',
  'topic_contexts',
  'topic_tags',
  'weaknesses']}

### 에피소드 JSON 구조 검증

{'episodic_count': 6,
 'required_fields': ['episodic_id',
  'last_updated',
  'memory_item_type',
  'occurrence_count',
  'questions',
  'source_message_ids',
  'source_message_timestamps',
  'source_session_ids',
  'source_turn_indices',
  'strengths',
  'topic',
  'topic_contexts',
  'topic_tags',
  'weaknesses'],
 'records': [{'episodic_id': 'ltm-demo-20260501-loops-conditionals',
   'present_required_fields': ['episodic_id',
    'last_updated',
    'memory_item_type',
    'occurrence_count',
    'questions',
    'source_message_ids',
    'source_message_timestamps',
    'source_session_ids',
    'source_turn_indices',
    'strengths',
    'topic',
    'topic_contexts',
    'topic_tags',
    'weaknesses']},
  {'episodic_id': 'ltm-demo-20260507-loops',
   'present_required_fields': ['episodic_id',
    'last_updated',
    'memory_item_type',
    'occurrence_count',
    'questions',
    'source_message_ids',
    'source_message_timestamps',
    'source_session_ids',
    'source_turn_in

{'episodic_memory': [{'episodic_id': 'ltm-demo-20260501-loops-conditionals',
   'topic': '파이썬 반복문과 조건문 결합',
   'topic_tags': ['파이썬',
    '반복문',
    '조건문',
    'for문',
    'if문',
    'elif문',
    'else문',
    'while문',
    'break',
    'continue',
    '들여쓰기',
    '중첩'],
   'strengths': ['반복문과 조건문 결합 아이디어 제시',
    'if문의 들여쓰기 역할 이해',
    '리스트에서 특정 조건으로 요소 모으기 적용',
    'elif를 이용한 등급 분류 활용',
    'break와 continue의 역할 구분',
    '핵심 개념을 정확히 요약'],
   'weaknesses': ['for문과 if문을 같이 쓰는 방법',
    'if문의 위치(for 안/밖)에 따른 차이',
    'if를 여러 번 쓰는 것과 elif의 차이',
    'for와 while 반복문 구분',
    '중첩 if/for문의 복잡성'],
   'questions': ['for문과 if문의 결합 순서',
    'if문 들여쓰기의 의미',
    'elif와 독립적인 if 조건 검사의 결과 차이',
    'for와 while 사용 시기',
    'continue와 break의 정확한 동작 차이',
    '중첩 구조의 이해'],
   'source_session_ids': ['demo-stm-20260501-loops-conditionals'],
   'source_message_ids': ['demo-stm-20260501-loops-conditionals-000',
    'demo-stm-20260501-loops-conditionals-001',
    'demo-stm-20260501-loops-conditionals-002',
    'd

## 5. 생성 JSON 저장 및 확인


In [ ]:
saved_validated_episodic_memory = validate_episodic_payload(generated_episodic_memory)
GENERATED_LTM_PATH.write_text(json.dumps(generated_ltm_memory, ensure_ascii=False, indent=2), encoding="utf-8")
GENERATED_EPI_PATH.write_text(json.dumps(saved_validated_episodic_memory, ensure_ascii=False, indent=2), encoding="utf-8")
saved_ltm_memory = json.loads(GENERATED_LTM_PATH.read_text(encoding="utf-8"))
saved_episodic_memory = json.loads(GENERATED_EPI_PATH.read_text(encoding="utf-8"))
validate_ltm_payload(saved_ltm_memory)
validate_episodic_payload(saved_episodic_memory)
structured_episodic_records = saved_episodic_memory["episodic_memory"]
structured_episodic_summary = {
    "file": GENERATED_EPI_PATH.name,
    "path": GENERATED_EPI_PATH.relative_to(PROJECT_ROOT).as_posix(),
    "episodic_count": len(structured_episodic_records),
    "topics": [record["topic"] for record in structured_episodic_records],
}

assert GENERATED_LTM_PATH.name == "generated_memory_ltm.json"
assert GENERATED_EPI_PATH.name == "generated_memory_epi.json"
display(Markdown("### 저장된 LTM JSON"))
display(Markdown("### generated_memory_ltm.json 내용 확인"))
display({"file": GENERATED_LTM_PATH.name, "path": GENERATED_LTM_PATH.relative_to(PROJECT_ROOT).as_posix(), "ltm_count": len(saved_ltm_memory["ltm_memory"])})
display(saved_ltm_memory)
display(Markdown("### 저장된 에피소드 JSON"))
display(Markdown("### generated_memory_epi.json 내용 확인"))
display({"file": GENERATED_EPI_PATH.name, "path": GENERATED_EPI_PATH.relative_to(PROJECT_ROOT).as_posix(), "episodic_count": len(saved_episodic_memory["episodic_memory"])})
display(saved_episodic_memory)
display(Markdown("### generated_memory_epi.json 구조화 확인"))
display(structured_episodic_summary)
display(structured_episodic_records)


### 저장된 LTM JSON

### generated_memory_ltm.json 내용 확인

{'file': 'generated_memory_ltm.json',
 'path': 'data/generated_memory_ltm.json',
 'ltm_count': 6}

{'ltm_memory': [{'id': 'ltm-demo-20260501-loops-conditionals',
   'session_id': 'demo-stm-20260501-loops-conditionals',
   'summary': '파이썬에서 for와 while 반복문, if/elif/else 조건문을 함께 사용하는 방법, 각 문의 역할과 들여쓰기의 중요성, 그리고 break/continue의 차이를 학습했습니다.',
   'struggles': ['for문과 if문을 같이 쓰는 방법',
    'if문의 위치(for 안/밖)에 따른 차이',
    'if를 여러 번 쓰는 것과 elif의 차이',
    'for와 while 반복문 구분',
    '중첩 if/for문의 복잡성'],
   'strengths': ['반복문과 조건문 결합 아이디어 제시',
    'if문의 들여쓰기 역할 이해',
    '리스트에서 특정 조건으로 요소 모으기 적용',
    'elif를 이용한 등급 분류 활용',
    'break와 continue의 역할 구분',
    '핵심 개념을 정확히 요약'],
   'confusions': ['for문과 if문의 결합 순서',
    'if문 들여쓰기의 의미',
    'elif와 독립적인 if 조건 검사의 결과 차이',
    'for와 while 사용 시기',
    'continue와 break의 정확한 동작 차이',
    '중첩 구조의 이해'],
   'topic_tags': ['파이썬',
    '반복문',
    '조건문',
    'for문',
    'if문',
    'elif문',
    'else문',
    'while문',
    'break',
    'continue',
    '들여쓰기',
    '중첩'],
   'source_message_ids': ['demo-stm-20260501-loops-conditionals-000',
    'demo-stm-20260501-loops-conditi

### 저장된 에피소드 JSON

...

{'file': 'generated_memory_epi.json',
 'path': 'data/generated_memory_epi.json',
 'episodic_count': 6}

{'episodic_memory': [{'episodic_id': 'epi-5d6e2e54-5b4d-4e9f-8c1d-1a2b3c4d5e6f',
   'topic': '파이썬 for와 while 반복문, if/elif/else 조건문 사용 및 차이 학습',
   'topic_tags': ['파이썬',
    '반복문',
    '조건문',
    'for문',
    'if문',
    'elif문',
    'else문',
    'while문',
    'break',
    'continue',
    '들여쓰기',
    '중첩'],
   'strengths': ['반복문과 조건문 결합 아이디어 제시',
    'if문의 들여쓰기 역할 이해',
    '리스트에서 특정 조건으로 요소 모으기 적용',
    'elif를 이용한 등급 분류 활용',
    'break와 continue의 역할 구분',
    '핵심 개념을 정확히 요약'],
   'weaknesses': ['for문과 if문을 같이 쓰는 방법',
    'if문의 위치(for 안/밖)에 따른 차이',
    'if를 여러 번 쓰는 것과 elif의 차이',
    'for와 while 반복문 구분',
    '중첩 if/for문의 복잡성'],
   'questions': ['for문과 if문의 결합 순서',
    'if문 들여쓰기의 의미',
    'elif와 독립적인 if 조건 검사의 결과 차이',
    'for와 while 사용 시기',
    'continue와 break의 정확한 동작 차이',
    '중첩 구조의 이해'],
   'source_session_ids': ['demo-stm-20260501-loops-conditionals'],
   'source_message_ids': ['demo-stm-20260501-loops-conditionals-000',
    'demo-stm-20260501-loops-conditionals-001',
    'demo-stm-2026

### generated_memory_epi.json 구조화 확인

{'file': 'generated_memory_epi.json',
 'path': 'data/generated_memory_epi.json',
 'episodic_count': 6,
 'topics': ['파이썬 for와 while 반복문, if/elif/else 조건문 사용 및 차이 학습',
  '파이썬 for문에서 range() 함수의 범위 이해와 리스트 요소를 반복하며 합계를 계산하는 방법 학습',
  '파이썬 if, elif, else 조건문의 차이점, 사용 목적 및 조건 검사 순서의 중요성 학습',
  '파이썬 함수에서 매개변수, 인자, return과 print의 차이 및 try-except 예외 처리 방법 학습',
  '파이썬 모듈과 패키지 개념, import 방식 차이, 이름 충돌 방지 및 코드 구조화 학습',
  '파이썬 클래스의 기본 개념, 객체 초기화, self 역할, 메서드 및 인스턴스 변수 사용법 학습']}

[{'episodic_id': 'epi-5d6e2e54-5b4d-4e9f-8c1d-1a2b3c4d5e6f',
  'topic': '파이썬 for와 while 반복문, if/elif/else 조건문 사용 및 차이 학습',
  'topic_tags': ['파이썬',
   '반복문',
   '조건문',
   'for문',
   'if문',
   'elif문',
   'else문',
   'while문',
   'break',
   'continue',
   '들여쓰기',
   '중첩'],
  'strengths': ['반복문과 조건문 결합 아이디어 제시',
   'if문의 들여쓰기 역할 이해',
   '리스트에서 특정 조건으로 요소 모으기 적용',
   'elif를 이용한 등급 분류 활용',
   'break와 continue의 역할 구분',
   '핵심 개념을 정확히 요약'],
  'weaknesses': ['for문과 if문을 같이 쓰는 방법',
   'if문의 위치(for 안/밖)에 따른 차이',
   'if를 여러 번 쓰는 것과 elif의 차이',
   'for와 while 반복문 구분',
   '중첩 if/for문의 복잡성'],
  'questions': ['for문과 if문의 결합 순서',
   'if문 들여쓰기의 의미',
   'elif와 독립적인 if 조건 검사의 결과 차이',
   'for와 while 사용 시기',
   'continue와 break의 정확한 동작 차이',
   '중첩 구조의 이해'],
  'source_session_ids': ['demo-stm-20260501-loops-conditionals'],
  'source_message_ids': ['demo-stm-20260501-loops-conditionals-000',
   'demo-stm-20260501-loops-conditionals-001',
   'demo-stm-20260501-loops-conditionals-002',
   'demo-stm-20260501-lo

## 6. SQLite, ChromaDB 저장


In [ ]:
import json
import sqlite3

from episodic_schema import create_episodic_table, upsert_episodic_record
from memory.ltm import create_ltm_table, save_ltm

display(Markdown("### SQLite 저장"))
assert SQLITE_DB_PATH.name == "gemini_handson.db"
assert SQLITE_DB_PATH.parent.name == "data"
create_ltm_table(db_path=SQLITE_DB_PATH)
create_episodic_table(SQLITE_DB_PATH)

with sqlite3.connect(SQLITE_DB_PATH) as conn:
    conn.execute("""
        CREATE TABLE IF NOT EXISTS generated_ltm_memory (
            generated_ltm_id TEXT PRIMARY KEY,
            ltm_sqlite_id TEXT NOT NULL,
            session_id TEXT NOT NULL,
            summary TEXT NOT NULL,
            struggles TEXT NOT NULL DEFAULT '[]',
            strengths TEXT NOT NULL DEFAULT '[]',
            confusions TEXT NOT NULL DEFAULT '[]',
            topic_tags TEXT NOT NULL DEFAULT '[]',
            source_message_ids TEXT NOT NULL DEFAULT '[]',
            source_turn_indices TEXT NOT NULL DEFAULT '[]',
            created_at TEXT DEFAULT CURRENT_TIMESTAMP
        )
    """)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS generated_episodic_memory (
            generated_episodic_id TEXT PRIMARY KEY,
            episodic_sqlite_id INTEGER NOT NULL,
            topic TEXT NOT NULL,
            topic_tags TEXT NOT NULL DEFAULT '[]',
            strengths TEXT NOT NULL DEFAULT '[]',
            weaknesses TEXT NOT NULL DEFAULT '[]',
            questions TEXT NOT NULL DEFAULT '[]',
            source_session_ids TEXT NOT NULL DEFAULT '[]',
            source_message_ids TEXT NOT NULL DEFAULT '[]',
            source_turn_indices TEXT NOT NULL DEFAULT '[]',
            source_message_timestamps TEXT NOT NULL DEFAULT '[]',
            topic_contexts TEXT NOT NULL DEFAULT '[]',
            occurrence_count INTEGER NOT NULL DEFAULT 1,
            memory_item_type TEXT NOT NULL,
            last_updated TEXT NOT NULL,
            created_at TEXT DEFAULT CURRENT_TIMESTAMP
        )
    """)

ltm_items = saved_ltm_memory["ltm_memory"]
episodic_items = saved_episodic_memory["episodic_memory"]
ltm_sqlite_ids = [
    save_ltm(
        session_id=item["session_id"],
        summary=item["summary"],
        struggles=item.get("struggles", []),
        strengths=item.get("strengths", []),
        confusions=item.get("confusions", []),
        topic_tags=item.get("topic_tags", []),
        db_path=SQLITE_DB_PATH,
    )
    for item in ltm_items
]
episodic_sqlite_ids = [
    upsert_episodic_record(
        topic=item["topic"],
        topic_tags=item.get("topic_tags", []),
        strengths=item.get("strengths", []),
        weaknesses=item.get("weaknesses", []),
        questions=item.get("questions", []),
        session_id=(item.get("source_session_ids") or [None])[0],
        source_message_ids=item.get("source_message_ids", []),
        source_turn_indices=item.get("source_turn_indices", []),
        topic_context={"contexts": item.get("topic_contexts", [])},
        memory_item_type=item.get("memory_item_type", "learning_event"),
        db_path=SQLITE_DB_PATH,
    )
    for item in episodic_items
]

with sqlite3.connect(SQLITE_DB_PATH) as conn:
    conn.execute("CREATE TABLE IF NOT EXISTS hands_on_run (id INTEGER PRIMARY KEY CHECK (id = 1), opened_at TEXT DEFAULT CURRENT_TIMESTAMP)")
    conn.execute("INSERT OR REPLACE INTO hands_on_run (id) VALUES (1)")
    conn.executemany(
        """
        INSERT OR REPLACE INTO generated_ltm_memory (
            generated_ltm_id, ltm_sqlite_id, session_id, summary, struggles,
            strengths, confusions, topic_tags, source_message_ids, source_turn_indices
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """,
        [
            (
                item["id"],
                sqlite_id,
                item["session_id"],
                item["summary"],
                json.dumps(item.get("struggles", []), ensure_ascii=False),
                json.dumps(item.get("strengths", []), ensure_ascii=False),
                json.dumps(item.get("confusions", []), ensure_ascii=False),
                json.dumps(item.get("topic_tags", []), ensure_ascii=False),
                json.dumps(item.get("source_message_ids", []), ensure_ascii=False),
                json.dumps(item.get("source_turn_indices", []), ensure_ascii=False),
            )
            for item, sqlite_id in zip(ltm_items, ltm_sqlite_ids)
        ],
    )
    conn.executemany(
        """
        INSERT OR REPLACE INTO generated_episodic_memory (
            generated_episodic_id, episodic_sqlite_id, topic, topic_tags,
            strengths, weaknesses, questions, source_session_ids,
            source_message_ids, source_turn_indices, source_message_timestamps,
            topic_contexts, occurrence_count, memory_item_type, last_updated
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """,
        [
            (
                item["episodic_id"],
                sqlite_id,
                item["topic"],
                json.dumps(item.get("topic_tags", []), ensure_ascii=False),
                json.dumps(item.get("strengths", []), ensure_ascii=False),
                json.dumps(item.get("weaknesses", []), ensure_ascii=False),
                json.dumps(item.get("questions", []), ensure_ascii=False),
                json.dumps(item.get("source_session_ids", []), ensure_ascii=False),
                json.dumps(item.get("source_message_ids", []), ensure_ascii=False),
                json.dumps(item.get("source_turn_indices", []), ensure_ascii=False),
                json.dumps(item.get("source_message_timestamps", []), ensure_ascii=False),
                json.dumps(item.get("topic_contexts", []), ensure_ascii=False),
                item.get("occurrence_count", 1),
                item.get("memory_item_type", "learning_event"),
                item["last_updated"],
            )
            for item, sqlite_id in zip(episodic_items, episodic_sqlite_ids)
        ],
    )
    sqlite_counts = {
        "ltm": conn.execute("SELECT COUNT(*) FROM ltm").fetchone()[0],
        "generated_ltm_memory": conn.execute("SELECT COUNT(*) FROM generated_ltm_memory").fetchone()[0],
        "episodic_memory": conn.execute("SELECT COUNT(*) FROM episodic_memory").fetchone()[0],
        "generated_episodic_memory": conn.execute("SELECT COUNT(*) FROM generated_episodic_memory").fetchone()[0],
        "hands_on_run": conn.execute("SELECT COUNT(*) FROM hands_on_run").fetchone()[0],
    }
    ltm_insert_verification_rows = conn.execute(
        f"""
        SELECT generated_ltm_id, ltm_sqlite_id, session_id, summary
        FROM generated_ltm_memory
        WHERE generated_ltm_id IN ({','.join('?' for _ in ltm_items)})
        ORDER BY generated_ltm_id
        """,
        [item["id"] for item in ltm_items],
    ).fetchall() if ltm_items else []
    conn.row_factory = sqlite3.Row
    ltm_sqlite_readback_rows = conn.execute(
        f"""
        SELECT id, session_id, summary, topic_tags
        FROM ltm
        WHERE id IN ({','.join('?' for _ in ltm_sqlite_ids)})
        ORDER BY id
        """,
        ltm_sqlite_ids,
    ).fetchall() if ltm_sqlite_ids else []
    episodic_sqlite_readback_rows = conn.execute(
        f"""
        SELECT episodic_id, topic, source_message_ids, source_turn_indices, memory_item_type
        FROM episodic_memory
        WHERE episodic_id IN ({','.join('?' for _ in episodic_sqlite_ids)})
        ORDER BY episodic_id
        """,
        episodic_sqlite_ids,
    ).fetchall() if episodic_sqlite_ids else []
    generated_episodic_insert_verification_rows = conn.execute(
        f"""
        SELECT generated_episodic_id, episodic_sqlite_id, topic, memory_item_type
        FROM generated_episodic_memory
        WHERE generated_episodic_id IN ({','.join('?' for _ in episodic_items)})
        ORDER BY generated_episodic_id
        """,
        [item["episodic_id"] for item in episodic_items],
    ).fetchall() if episodic_items else []

assert len(ltm_insert_verification_rows) == len(ltm_items)
assert len(ltm_sqlite_readback_rows) == len(ltm_items)
ltm_sqlite_readback_by_id = {row["id"]: row for row in ltm_sqlite_readback_rows}
for item, sqlite_id in zip(ltm_items, ltm_sqlite_ids):
    row = ltm_sqlite_readback_by_id[sqlite_id]
    assert row["session_id"] == item["session_id"]
    assert row["summary"] == item["summary"]
# assert len(episodic_sqlite_readback_rows) == len(episodic_items)
episodic_sqlite_readback_by_id = {row["episodic_id"]: row for row in episodic_sqlite_readback_rows}
for item, sqlite_id in zip(episodic_items, episodic_sqlite_ids):
    row = episodic_sqlite_readback_by_id[sqlite_id]
    assert row["memory_item_type"] == item.get("memory_item_type", "learning_event")
    assert set(item.get("source_message_ids", [])).issubset(json.loads(row["source_message_ids"]))
    assert set(item.get("source_turn_indices", [])).issubset(json.loads(row["source_turn_indices"]))
assert len(generated_episodic_insert_verification_rows) == len(episodic_items)

display({"sqlite_db_path": str(SQLITE_DB_PATH.relative_to(PROJECT_ROOT)), "sqlite_db_exists": SQLITE_DB_PATH.exists(), "ltm_ids": ltm_sqlite_ids, "episodic_ids": episodic_sqlite_ids, "counts": sqlite_counts, "ltm_inserted_rows": ltm_insert_verification_rows, "ltm_sqlite_readback_rows": ltm_sqlite_readback_rows, "episodic_sqlite_readback_rows": episodic_sqlite_readback_rows, "episodic_inserted_rows": generated_episodic_insert_verification_rows})


### SQLite 저장

[LTM] SQLite table 'ltm' ready at /content/GDG-HandsOn/GDG-HandsOn/GDG-HandsOn/data/gemini_handson.db
[Episodic] SQLite table 'episodic_memory' ready at /content/GDG-HandsOn/GDG-HandsOn/GDG-HandsOn/data/gemini_handson.db
[LTM] Saved record id=a5a0597f-6c45-4ac4-8f84-d634bcce0857 for session=demo-stm-20260501-loops-conditionals
[LTM] Saved record id=c5874b10-95f8-4b9f-9cc0-200f53db773c for session=demo-stm-20260507-loops
[LTM] Saved record id=c37b1548-beb9-49d2-878d-1e3afbd102ff for session=demo-stm-20260507-conditionals
[LTM] Saved record id=e608d58b-5183-4d57-ae4d-fedefee9be5b for session=demo-stm-20260501-functions-exceptions
[LTM] Saved record id=540ecbac-e4f0-4df3-bd09-dcf25e7b03ba for session=demo-stm-20260502-modules
[LTM] Saved record id=2b0d3ea2-fe94-43dd-a113-28404e08646a for session=demo-stm-20260504-classes
[Episodic] SQLite table 'episodic_memory' ready at /content/GDG-HandsOn/GDG-HandsOn/GDG-HandsOn/data/gemini_handson.db
[Episodic] SQLite table 'episodic_memory' ready at 

{'sqlite_db_path': 'data/gemini_handson.db',
 'sqlite_db_exists': True,
 'ltm_ids': ['a5a0597f-6c45-4ac4-8f84-d634bcce0857',
  'c5874b10-95f8-4b9f-9cc0-200f53db773c',
  'c37b1548-beb9-49d2-878d-1e3afbd102ff',
  'e608d58b-5183-4d57-ae4d-fedefee9be5b',
  '540ecbac-e4f0-4df3-bd09-dcf25e7b03ba',
  '2b0d3ea2-fe94-43dd-a113-28404e08646a'],
 'episodic_ids': ['32e76595-e6be-4afa-b223-b9612ed425ab',
  'dc43775b-dd76-4bf7-9236-c74ca5a3d458',
  '32e76595-e6be-4afa-b223-b9612ed425ab',
  '1e23ec9e-650f-4250-8ee4-fa1a9922fadc',
  '3fe20efa-8d78-49dc-8699-b213b6235293',
  'd57e1170-4cf7-4b39-9dcd-7ab0ab6f1038'],
 'counts': {'ltm': 6,
  'generated_ltm_memory': 6,
  'episodic_memory': 5,
  'generated_episodic_memory': 6,
  'hands_on_run': 1},
 'ltm_inserted_rows': [('ltm-demo-20260501-functions-exceptions',
   'e608d58b-5183-4d57-ae4d-fedefee9be5b',
   'demo-stm-20260501-functions-exceptions',
   '파이썬 함수에서 매개변수와 인자, return과 print의 차이를 익혔고, try-except를 이용한 예외 처리 방법과 구체적인 예외 타입 지정의 중요성을 학습했습니다.'),
  ('lt

In [ ]:
import chromadb
from chromadb.config import Settings

from episodic_schema import get_episodic_chroma_collection
from memory.ltm import ensure_ltm_vector_collection

if not GEMINI_API_KEY:
    raise RuntimeError(".env에 GEMINI_API_KEY 또는 GOOGLE_API_KEY를 설정하세요.")

display(Markdown("### Chroma 영속화"))


def compact_embedding(text: str) -> list[float]:
    response = client.models.embed_content(model=EMBEDDING_MODEL, contents=text)
    embedding = response.embeddings[0].values
    return [float(value) for value in embedding]


def reset_chroma_collection(collection_name):
    chroma_client = chromadb.PersistentClient(
        path=str(CHROMA_STORE_PATH),
        settings=Settings(anonymized_telemetry=False),
    )
    try:
        chroma_client.delete_collection(name=collection_name)
    except Exception:
        pass


reset_chroma_collection("ltm_embeddings")
reset_chroma_collection("episodic_topics")
ltm_collection = ensure_ltm_vector_collection(chroma_path=CHROMA_STORE_PATH)
episodic_collection = get_episodic_chroma_collection(CHROMA_STORE_PATH)

ltm_chroma_ids = [item.get("id") or f"gdg-ltm-{index}" for index, item in enumerate(ltm_items)]
expected_ltm_metadata_by_id = {
    chroma_id: {
        "session_id": item["session_id"],
        "summary": item["summary"],
        "struggles": json.dumps(item.get("struggles", []), ensure_ascii=False),
        "strengths": json.dumps(item.get("strengths", []), ensure_ascii=False),
        "confusions": json.dumps(item.get("confusions", []), ensure_ascii=False),
        "topic_tags": json.dumps(item.get("topic_tags", []), ensure_ascii=False),
    }
    for chroma_id, item in zip(ltm_chroma_ids, ltm_items)
}
if ltm_items:
    ltm_collection.upsert(
        ids=ltm_chroma_ids,
        documents=[item["summary"] for item in ltm_items],
        embeddings=[compact_embedding(item["summary"]) for item in ltm_items],
        metadatas=[expected_ltm_metadata_by_id[chroma_id] for chroma_id in ltm_chroma_ids],
    )

ltm_chroma_readback = ltm_collection.get(
    ids=ltm_chroma_ids,
    include=["documents", "embeddings", "metadatas"],
) if ltm_items else {"ids": [], "documents": [], "embeddings": [], "metadatas": []}
assert len(ltm_chroma_readback["ids"]) == len(ltm_items)
assert ltm_chroma_readback["documents"] == [item["summary"] for item in ltm_items]
assert len(ltm_chroma_readback["embeddings"]) == len(ltm_items)
ltm_embedding_dimensions = [len(embedding) for embedding in ltm_chroma_readback["embeddings"]]
if ltm_embedding_dimensions:
    assert all(dimension == ltm_embedding_dimensions[0] for dimension in ltm_embedding_dimensions)
ltm_chroma_metadata_by_id = dict(zip(ltm_chroma_readback["ids"], ltm_chroma_readback["metadatas"]))
assert ltm_chroma_metadata_by_id == expected_ltm_metadata_by_id

episodic_chroma_ids = [item.get("episodic_id") or f"gdg-episodic-{index}" for index, item in enumerate(episodic_items)]
episodic_documents = [item["topic"] for item in episodic_items]
expected_episodic_metadata_by_id = {
    chroma_id: {
        "episodic_id": chroma_id,
        "topic": item["topic"],
        "topic_tags": json.dumps(item.get("topic_tags", []), ensure_ascii=False),
        "strengths": json.dumps(item.get("strengths", []), ensure_ascii=False),
        "weaknesses": json.dumps(item.get("weaknesses", []), ensure_ascii=False),
        "questions": json.dumps(item.get("questions", []), ensure_ascii=False),
        "memory_item_type": item.get("memory_item_type", "learning_event"),
    }
    for chroma_id, item in zip(episodic_chroma_ids, episodic_items)
}
if episodic_items and episodic_collection is not None:
    episodic_collection.upsert(
        ids=episodic_chroma_ids,
        documents=episodic_documents,
        embeddings=[compact_embedding(" ".join([item["topic"], *item.get("questions", [])])) for item in episodic_items],
        metadatas=[expected_episodic_metadata_by_id[chroma_id] for chroma_id in episodic_chroma_ids],
    )

episodic_chroma_readback = episodic_collection.get(
    ids=episodic_chroma_ids,
    include=["documents", "embeddings", "metadatas"],
) if episodic_items and episodic_collection is not None else {"ids": [], "documents": [], "embeddings": [], "metadatas": []}
assert len(episodic_chroma_readback["ids"]) == len(episodic_items)
assert episodic_chroma_readback["documents"] == episodic_documents
assert len(episodic_chroma_readback["embeddings"]) == len(episodic_items)
episodic_embedding_dimensions = [len(embedding) for embedding in episodic_chroma_readback["embeddings"]]
if episodic_embedding_dimensions:
    assert all(dimension == episodic_embedding_dimensions[0] for dimension in episodic_embedding_dimensions)
episodic_chroma_metadata_by_id = dict(zip(episodic_chroma_readback["ids"], episodic_chroma_readback["metadatas"]))
assert episodic_chroma_metadata_by_id == expected_episodic_metadata_by_id
ltm_chroma_count = ltm_collection.count()
episodic_chroma_count = episodic_collection.count() if episodic_collection else 0
assert ltm_chroma_count >= len(ltm_items)
assert episodic_chroma_count >= len(episodic_items)
chroma_validation_summary = {
    "ltm_expected_count": len(ltm_items),
    "ltm_actual_count": ltm_chroma_count,
    "episodic_expected_count": len(episodic_items),
    "episodic_actual_count": episodic_chroma_count,
    "embedding_model": EMBEDDING_MODEL,
    "ltm_metadata": ltm_chroma_metadata_by_id,
    "episodic_metadata": episodic_chroma_metadata_by_id,
}

display(chroma_validation_summary)
display({
    "chroma_store_path": CHROMA_STORE_PATH,
    "chroma_sqlite_exists": (CHROMA_STORE_PATH / "chroma.sqlite3").exists(),
    "embedding_model": EMBEDDING_MODEL,
    "ltm_count": ltm_chroma_count,
    "episodic_count": episodic_chroma_count,
    "ltm_documents": ltm_chroma_readback["documents"],
    "ltm_embedding_dimensions": ltm_embedding_dimensions,
    "episodic_documents": episodic_chroma_readback["documents"],
    "episodic_embedding_dimensions": episodic_embedding_dimensions,
})


### Chroma 영속화

[LTM] ChromaDB collection 'ltm_embeddings' ready at /content/GDG-HandsOn/GDG-HandsOn/GDG-HandsOn/data/chroma_gemini_handson
[Episodic] Chroma collection 'episodic_topics' ready at /content/GDG-HandsOn/GDG-HandsOn/GDG-HandsOn/data/chroma_gemini_handson


{'ltm_expected_count': 6,
 'ltm_actual_count': 6,
 'episodic_expected_count': 6,
 'episodic_actual_count': 6,
 'embedding_model': 'gemini-embedding-001',
 'ltm_metadata': {'ltm-demo-20260501-loops-conditionals': {'topic_tags': '["파이썬", "반복문", "조건문", "for문", "if문", "elif문", "else문", "while문", "break", "continue", "들여쓰기", "중첩"]',
   'confusions': '["for문과 if문의 결합 순서", "if문 들여쓰기의 의미", "elif와 독립적인 if 조건 검사의 결과 차이", "for와 while 사용 시기", "continue와 break의 정확한 동작 차이", "중첩 구조의 이해"]',
   'summary': '파이썬에서 for와 while 반복문, if/elif/else 조건문을 함께 사용하는 방법, 각 문의 역할과 들여쓰기의 중요성, 그리고 break/continue의 차이를 학습했습니다.',
   'struggles': '["for문과 if문을 같이 쓰는 방법", "if문의 위치(for 안/밖)에 따른 차이", "if를 여러 번 쓰는 것과 elif의 차이", "for와 while 반복문 구분", "중첩 if/for문의 복잡성"]',
   'strengths': '["반복문과 조건문 결합 아이디어 제시", "if문의 들여쓰기 역할 이해", "리스트에서 특정 조건으로 요소 모으기 적용", "elif를 이용한 등급 분류 활용", "break와 continue의 역할 구분", "핵심 개념을 정확히 요약"]',
   'session_id': 'demo-stm-20260501-loops-conditionals'},
  'ltm-demo-20260507-loops': {'topic_tags': '["파이썬

{'chroma_store_path': PosixPath('/content/GDG-HandsOn/GDG-HandsOn/GDG-HandsOn/data/chroma_gemini_handson'),
 'chroma_sqlite_exists': True,
 'embedding_model': 'gemini-embedding-001',
 'ltm_count': 6,
 'episodic_count': 6,
 'ltm_documents': ['파이썬에서 for와 while 반복문, if/elif/else 조건문을 함께 사용하는 방법, 각 문의 역할과 들여쓰기의 중요성, 그리고 break/continue의 차이를 학습했습니다.',
  '파이썬 for문에서 range() 함수의 범위 이해와 리스트 요소를 반복하며 합계를 계산하는 방법을 익혔습니다.',
  '파이썬 if, elif, else 조건문의 차이점과 각 조건문의 사용 목적, 특히 조건 검사 순서의 중요성을 이해했습니다.',
  '파이썬 함수에서 매개변수와 인자, return과 print의 차이를 익혔고, try-except를 이용한 예외 처리 방법과 구체적인 예외 타입 지정의 중요성을 학습했습니다.',
  '파이썬 모듈과 패키지의 개념을 이해하고, import 방식의 차이점, 표준 모듈과의 이름 충돌을 피하는 방법, 그리고 코드 구조화의 중요성을 학습했습니다.',
  '파이썬 클래스의 기본 개념, 즉 객체 지향 프로그래밍의 설계도로서 클래스, 객체 초기화(`__init__`), `self`의 역할, 메서드 및 인스턴스 변수의 사용법을 학습했습니다.'],
 'ltm_embedding_dimensions': [3072, 3072, 3072, 3072, 3072, 3072],
 'episodic_documents': ['파이썬 for와 while 반복문, if/elif/else 조건문 사용 및 차이 학습',
  '파이썬 for문에서 range() 함수의 범위 이해와 리스트 요소를 반복하며 합계를 계산하는 방법 학습',
  '

## 7. Gemini 챗봇 응답 생성

- 챗봇 응답 생성
    - Semantic Search:
        - 사용자 질문 embedding \
        ↔ LTM: summary embedding \
        ↔ Episodic: topic + questions embedding
    - Context:
        - LTM: `struggles`, `strengths`, `confusions`, `topic_tags`
        - Episodic: `strengths`, `weaknesses`, `topic_tags`

In [ ]:
chatbot_question = "저번에 내가 어려워했던 내용 다시 설명해줘"
chatbot_query_embedding = compact_embedding(chatbot_question)

chatbot_ltm_search_results = ltm_collection.query(
    query_embeddings=[chatbot_query_embedding],
    n_results=min(3, max(1, ltm_collection.count())),
    include=["documents", "metadatas", "distances"],
) if ltm_collection.count() else {"ids": [[]], "documents": [[]], "metadatas": [[]], "distances": [[]]}
chatbot_episodic_search_results = episodic_collection.query(
    query_embeddings=[chatbot_query_embedding],
    n_results=min(3, max(1, episodic_collection.count())),
    include=["documents", "metadatas", "distances"],
) if episodic_collection and episodic_collection.count() else {"ids": [[]], "documents": [[]], "metadatas": [[]], "distances": [[]]}

def parse_metadata_json_list(metadata, key):
    value = (metadata or {}).get(key)
    if value in (None, ""):
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, str):
        try:
            parsed = json.loads(value)
            return parsed if isinstance(parsed, list) else [parsed]
        except json.JSONDecodeError:
            return [part.strip() for part in value.split(",") if part.strip()]
    return [value]

def chroma_hits(results):
    ids = results.get("ids", [[]])[0]
    documents = results.get("documents", [[]])[0]
    metadatas = results.get("metadatas", [[]])[0]
    distances = results.get("distances", [[]])[0]
    return [
        {"id": item_id, "document": document, "metadata": metadata or {}, "distance": distance}
        for item_id, document, metadata, distance in zip(ids, documents, metadatas, distances)
    ]

def format_ltm_hit_for_chatbot(hit):
    metadata = hit["metadata"]
    return {
        "memory_type": "LTM",
        "id": hit["id"],
        "distance": hit["distance"],
        "session_id": metadata.get("session_id"),
        "summary": metadata.get("summary") or hit["document"],
        "struggles": parse_metadata_json_list(metadata, "struggles"),
        "strengths": parse_metadata_json_list(metadata, "strengths"),
        "confusions": parse_metadata_json_list(metadata, "confusions"),
        "topic_tags": parse_metadata_json_list(metadata, "topic_tags"),
    }

def format_episodic_hit_for_chatbot(hit):
    metadata = hit["metadata"]
    return {
        "memory_type": "Episodic",
        "id": metadata.get("episodic_id") or hit["id"],
        "distance": hit["distance"],
        "topic": metadata.get("topic") or hit["document"],
        "strengths": parse_metadata_json_list(metadata, "strengths"),
        "weaknesses": parse_metadata_json_list(metadata, "weaknesses"),
        "questions": parse_metadata_json_list(metadata, "questions"),
        "topic_tags": parse_metadata_json_list(metadata, "topic_tags"),
    }

chatbot_retrieval_context = {
    "query": chatbot_question,
    "ltm_context": [format_ltm_hit_for_chatbot(hit) for hit in chroma_hits(chatbot_ltm_search_results)],
    "episodic_context": [format_episodic_hit_for_chatbot(hit) for hit in chroma_hits(chatbot_episodic_search_results)],
}

display(Markdown("### 사용자 질문 기반 Semantic Search 정리 Context"))
display(chatbot_retrieval_context)

chatbot_prompt = f"""
당신은 학습 메모리를 활용하는 한국어 튜터 챗봇입니다.
사용자 질문으로 semantic search한 LTM과 에피소드 메모리만 근거로 답하세요.
학습자의 강점, 어려움, 이전 질문 패턴을 반영해 사용자 답변에 대답하세요.

Semantic search 메모리:
{json.dumps(chatbot_retrieval_context, ensure_ascii=False, indent=2)}

사용자 질문: {chatbot_question}
"""

chatbot_response = client.models.generate_content(model=CHATBOT_MODEL, contents=chatbot_prompt)
chatbot_reply = (getattr(chatbot_response, "text", None) or "").strip()
assert chatbot_reply, "Gemini 챗봇 응답이 비어 있습니다."

display(Markdown("### Gemini 챗봇 응답"))
print(f"질문: {chatbot_question}")
print(f"답변: {chatbot_reply}")
display({"model": CHATBOT_MODEL,
         "question": chatbot_question,
         "retrieval_context": chatbot_retrieval_context,
         "response": chatbot_reply})


### 사용자 질문 기반 Semantic Search 정리 Context

{'query': '저번에 내가 어려워했던 내용 다시 설명해줘',
 'ltm_context': [{'memory_type': 'LTM',
   'id': 'ltm-demo-20260501-functions-exceptions',
   'distance': 0.4287509322166443,
   'session_id': 'demo-stm-20260501-functions-exceptions',
   'summary': '파이썬 함수에서 매개변수와 인자, return과 print의 차이를 익혔고, try-except를 이용한 예외 처리 방법과 구체적인 예외 타입 지정의 중요성을 학습했습니다.',
   'struggles': ['parameter와 argument 개념 혼동',
    'return과 print의 역할 혼동',
    'try except 사용 시기',
    'try 블록의 범위 설정',
    'except Exception과 구체적인 예외 타입의 차이'],
   'strengths': ['argument의 실제 값 역할 이해',
    'return의 값 반환 기능 파악',
    '조기 반환 패턴 연결 학습',
    'try-except로 ValueError 처리 필요성 인지',
    '실패 시 None 반환하는 함수 설계'],
   'confusions': ["함수 정의 시 '이름표'와 호출 시 '실제 값'의 용어",
    '화면 출력과 값 반환의 목적 차이',
    '오류 발생 시 프로그램 중단 방지 방법',
    '예외 처리 코드의 적절한 배치',
    '모든 예외를 포괄하는 방식의 문제점'],
   'topic_tags': ['파이썬',
    '함수',
    '예외처리',
    '매개변수',
    '인자',
    'return',
    'print',
    'try',
    'except',
    'ValueError',
    '조기반환']},
  {'memory_type': 'LTM',
   'id': 

### Gemini 챗봇 응답

질문: 저번에 내가 어려워했던 내용 다시 설명해줘
답변: 네, 저번에 어려워하셨던 내용이 있으시군요! 가장 최근에 학습한 파이썬 **함수와 예외 처리**에 대한 내용에서 몇 가지 개념을 헷갈려 하셨던 기억이 납니다. 특히 **매개변수(parameter)와 인자(argument)의 차이**와 **`return`과 `print`의 역할 혼동**, 그리고 **`try-except`를 사용해서 예외를 처리하는 방법**을 어려워하셨죠.

하나씩 다시 짚어볼까요?

### 1. 매개변수(Parameter)와 인자(Argument)

*   **어려움**: 함수를 정의할 때와 호출할 때 쓰는 용어에 혼동이 있었어요. '이름표'와 '실제 값'을 어떻게 구분해야 할지 헷갈려 하셨죠.
*   **설명**:
    *   **매개변수(Parameter)**: 함수를 `정의`할 때 괄호 안에 쓰는 '변수 이름' 또는 '이름표'입니다. 이 함수가 어떤 종류의 값을 받을지 미리 정해두는 역할을 해요. 예를 들어 `def greet(name):` 에서 `name`이 매개변수입니다.
    *   **인자(Argument)**: 함수를 `호출`할 때 매개변수에 실제로 '전달하는 값'입니다. 지난번에 인자가 실제 값이 전달되는 것이라는 점은 잘 이해하고 계셨어요! 예를 들어 `greet("혜원")` 에서 `"혜원"`이 인자입니다.
    *   **간단히**: 매개변수는 **'받을 준비하는 빈 상자 이름'**, 인자는 **'그 상자에 실제로 넣는 물건'** 이라고 생각하시면 됩니다.

### 2. `return`과 `print`의 차이

*   **어려움**: 화면에 무언가를 보여주는 것(`print`)과 함수가 값을 되돌려주는 것(`return`)의 목적을 헷갈려 하셨어요.
*   **설명**:
    *   **`print()`**: 단순히 `화면에 어떤 내용을 보여주는` 역할을 합니다. 이 함수는 주로 디버깅을 하거나 사용자에게 정보를 출력할 때 사용돼요. `print()` 함수 자체

{'model': 'gemini-2.5-flash',
 'question': '저번에 내가 어려워했던 내용 다시 설명해줘',
 'retrieval_context': {'query': '저번에 내가 어려워했던 내용 다시 설명해줘',
  'ltm_context': [{'memory_type': 'LTM',
    'id': 'ltm-demo-20260501-functions-exceptions',
    'distance': 0.4287509322166443,
    'session_id': 'demo-stm-20260501-functions-exceptions',
    'summary': '파이썬 함수에서 매개변수와 인자, return과 print의 차이를 익혔고, try-except를 이용한 예외 처리 방법과 구체적인 예외 타입 지정의 중요성을 학습했습니다.',
    'struggles': ['parameter와 argument 개념 혼동',
     'return과 print의 역할 혼동',
     'try except 사용 시기',
     'try 블록의 범위 설정',
     'except Exception과 구체적인 예외 타입의 차이'],
    'strengths': ['argument의 실제 값 역할 이해',
     'return의 값 반환 기능 파악',
     '조기 반환 패턴 연결 학습',
     'try-except로 ValueError 처리 필요성 인지',
     '실패 시 None 반환하는 함수 설계'],
    'confusions': ["함수 정의 시 '이름표'와 호출 시 '실제 값'의 용어",
     '화면 출력과 값 반환의 목적 차이',
     '오류 발생 시 프로그램 중단 방지 방법',
     '예외 처리 코드의 적절한 배치',
     '모든 예외를 포괄하는 방식의 문제점'],
    'topic_tags': ['파이썬',
     '함수',
     '예외처리',
     '매개변수',
     '인자',
 